# NumPy Lab: Sales Data Analysis
Two weeks of online-store revenue (tenge) for 3 product categories, analysed with NumPy.

In [1]:
import numpy as np

rng = np.random.default_rng(seed=2026)

# Sales for Week 1: 7 days, 3 product categories
week1 = rng.integers(low=500, high=2000, size=(7, 3))

# Sales for Week 2: 7 days, 3 product categories
week2 = rng.integers(low=500, high=2000, size=(7, 3))

# Product categories
categories = np.array(["Electronics", "Clothing", "Home"])

print("Week 1 sales:")
print(week1)

print("Week 2 sales:")
print(week2)

print("Categories:")
print(categories)

Week 1 sales:
[[1777  768  539]
 [1459 1048 1200]
 [ 619 1055 1465]
 [1032 1746 1685]
 [1556 1857 1581]
 [ 766 1787 1479]
 [ 647  947  748]]
Week 2 sales:
[[1950 1591 1879]
 [ 923 1453 1406]
 [1629  677 1272]
 [1466 1738 1484]
 [1172 1190 1008]
 [ 735  916  719]
 [ 839 1780 1288]]
Categories:
['Electronics' 'Clothing' 'Home']


## Block 1 — Create and Inspect Data
### Task 1.1 — Generate Sales Data
Both arrays were generated above from one generator with `seed=2026`. Now inspect them.

In [2]:
for name, arr in [("week1", week1), ("week2", week2)]:
    print(f"{name}: shape={arr.shape}, ndim={arr.ndim}, size={arr.size}, dtype={arr.dtype}")

week1: shape=(7, 3), ndim=2, size=21, dtype=int64
week2: shape=(7, 3), ndim=2, size=21, dtype=int64


### Task 1.2 — Understand the Data
- **Each row** is one day (7 rows = 7 days of a week).
- **Each column** is one product category: Electronics, Clothing, Home.
- **Why fix the seed?** The seed makes the "random" numbers reproducible: every run (and every reviewer) gets the same data, so results and conclusions can be checked and compared.

## Block 2 — Select and Filter Data
### Task 2.1 — Indexing and Slicing

In [3]:
print("First day:           ", week1[0])
print("Last day:            ", week1[-1])
print("Electronics, week:   ", week1[:, 0])
print("Clothing, first 3 d: ", week1[:3, 1])
print("Days 3 to 5 incl.:\n", week1[2:5])   # days are 1-based, indexes 2,3,4

First day:            [1777  768  539]
Last day:             [647 947 748]
Electronics, week:    [1777 1459  619 1032 1556  766  647]
Clothing, first 3 d:  [ 768 1048 1055]
Days 3 to 5 incl.:
 [[ 619 1055 1465]
 [1032 1746 1685]
 [1556 1857 1581]]


### Task 2.2 — Boolean Filtering

In [4]:
print("> 1500:          ", week1[week1 > 1500])
print("< 800:           ", week1[week1 < 800])
print("1000..1500 incl.:", week1[(week1 >= 1000) & (week1 <= 1500)])
print("Days with Electronics > 1200:", np.sum(week1[:, 0] > 1200))

> 1500:           [1777 1746 1685 1556 1857 1581 1787]
< 800:            [768 539 619 766 647 748]
1000..1500 incl.: [1459 1048 1200 1055 1465 1032 1479]
Days with Electronics > 1200: 3


**Conditions used:** a comparison (`week1 > 1500`) gives a Boolean array of the same shape; `week1[mask]` keeps only the `True` positions (as a flat 1-D array). For the range, two conditions are combined with `&`: `(week1 >= 1000) & (week1 <= 1500)`. The Electronics count sums the Boolean column (`True` = 1).

**Why `&` and `|` instead of `and` / `or`?** `and`/`or` need one single True/False for the whole object and raise an error for arrays. `&`/`|` work element by element. Each condition needs parentheses because `&`/`|` bind tighter than comparisons.

**Business problem:** Boolean filtering finds strong days (to repeat what worked), weak days (to run promotions), big orders, or unusual values/outliers to check for errors.

## Block 3 — Summarize Sales
### Task 3.1 — Aggregation

In [5]:
print("Total weekly revenue:   ", week1.sum())
print("Average daily revenue:  ", week1.sum(axis=1).mean())
print("Average per cell:       ", week1.mean())
print("Max / min single value: ", week1.max(), "/", week1.min())
print("Total per category:     ", dict(zip(categories, week1.sum(axis=0))))
print("Average revenue per day (mean of 3 categories):", week1.mean(axis=1))

Total weekly revenue:    25761
Average daily revenue:   3680.1428571428573
Average per cell:        1226.7142857142858
Max / min single value:  1857 / 539
Total per category:      {np.str_('Electronics'): np.int64(7856), np.str_('Clothing'): np.int64(9208), np.str_('Home'): np.int64(8697)}
Average revenue per day (mean of 3 categories): [1028.         1235.66666667 1046.33333333 1487.66666667 1664.66666667
 1344.          780.66666667]


`axis=0` collapses the rows (gives one value per **column / category**); `axis=1` collapses the columns (one value per **row / day**). "Average daily revenue" is taken on daily totals; "average revenue for each day" is the per-day mean over categories.

### Task 3.2 — Cumulative Sales

In [6]:
daily_total = week1.sum(axis=1)
cumulative = np.cumsum(daily_total)
print("Daily totals:      ", daily_total)
print("Cumulative revenue:", cumulative)

half = week1.sum() / 2
day = np.argmax(cumulative > half) + 1   # first True; +1 because days start at 1
print(f"Half of weekly revenue = {half}; exceeded after day {day} (cumulative {cumulative[day-1]})")

Daily totals:       [3084 3707 3139 4463 4994 4032 2342]
Cumulative revenue: [ 3084  6791  9930 14393 19387 23419 25761]
Half of weekly revenue = 12880.5; exceeded after day 4 (cumulative 14393)


**The last value** of the cumulative sum equals the total revenue of the whole week. Cumulative sums show how revenue builds up over time and when targets (like half the week's revenue) are reached.

## Block 4 — Transform Sales
### Task 4.1 — Vectorized Price Increase

In [7]:
new_week1 = week1.copy()           # copy, so the original stays unchanged
new_week1 = new_week1 * 1.12       # +12% for every value at once
print("Original:\n", week1)
print("New (+12%):\n", new_week1)
new_week1_int = np.round(new_week1).astype(int)
print("New, rounded to integers:\n", new_week1_int)

Original:
 [[1777  768  539]
 [1459 1048 1200]
 [ 619 1055 1465]
 [1032 1746 1685]
 [1556 1857 1581]
 [ 766 1787 1479]
 [ 647  947  748]]
New (+12%):
 [[1990.24  860.16  603.68]
 [1634.08 1173.76 1344.  ]
 [ 693.28 1181.6  1640.8 ]
 [1155.84 1955.52 1887.2 ]
 [1742.72 2079.84 1770.72]
 [ 857.92 2001.44 1656.48]
 [ 724.64 1060.64  837.76]]
New, rounded to integers:
 [[1990  860  604]
 [1634 1174 1344]
 [ 693 1182 1641]
 [1156 1956 1887]
 [1743 2080 1771]
 [ 858 2001 1656]
 [ 725 1061  838]]


**No `for` loop needed:** vectorization applies the operation to every element at once, in fast compiled code. It is shorter, easier to read and much faster on large data.

### Task 4.2 — Categorize Sales

In [8]:
mask = week1 > 1500
print("Mask:\n", mask)
labels = np.where(mask, "High", "Low")
print("Original:\n", week1)
print("Labels:\n", labels)

Mask:
 [[ True False False]
 [False False False]
 [False False False]
 [False  True  True]
 [ True  True  True]
 [False  True False]
 [False False False]]
Original:
 [[1777  768  539]
 [1459 1048 1200]
 [ 619 1055 1465]
 [1032 1746 1685]
 [1556 1857 1581]
 [ 766 1787 1479]
 [ 647  947  748]]
Labels:
 [['High' 'Low' 'Low']
 ['Low' 'Low' 'Low']
 ['Low' 'Low' 'Low']
 ['Low' 'High' 'High']
 ['High' 'High' 'High']
 ['Low' 'High' 'Low']
 ['Low' 'Low' 'Low']]


`np.where(condition, a, b)` checks the condition for every element and returns `a` where it is `True` and `b` where it is `False`, producing a new array of the same shape.

## Block 5 — Reshape and Combine
### Task 5.1 — Reshape Sales Data

In [9]:
flat = week1.reshape(-1)  # 1-D view of all values
print(flat, "| elements:", flat.size)
back = flat.reshape(7, 3)
print(back, back.shape, "| equal to original:", np.array_equal(back, week1))

[1777  768  539 1459 1048 1200  619 1055 1465 1032 1746 1685 1556 1857
 1581  766 1787 1479  647  947  748] | elements: 21
[[1777  768  539]
 [1459 1048 1200]
 [ 619 1055 1465]
 [1032 1746 1685]
 [1556 1857 1581]
 [ 766 1787 1479]
 [ 647  947  748]] (7, 3) | equal to original: True


The element count (21) does not change because reshape only changes how the same values are *arranged* (the shape), it never adds or removes data.

### Task 5.2 — Use -1 in Reshape

In [10]:
a = week1.reshape(7, -1)
b = week1.reshape(-1, 3)
print(a, a.shape)
print(b, b.shape)

[[1777  768  539]
 [1459 1048 1200]
 [ 619 1055 1465]
 [1032 1746 1685]
 [1556 1857 1581]
 [ 766 1787 1479]
 [ 647  947  748]] (7, 3)
[[1777  768  539]
 [1459 1048 1200]
 [ 619 1055 1465]
 [1032 1746 1685]
 [1556 1857 1581]
 [ 766 1787 1479]
 [ 647  947  748]] (7, 3)


`-1` means "work it out". NumPy divides the total number of elements by the known dimension: for `(7, -1)` it is 21 / 7 = 3 columns; for `(-1, 3)` it is 21 / 3 = 7 rows. Both give `(7, 3)`.

### Task 5.3 — Understand Non-Even Reshaping

In [11]:
demo = np.arange(1, 21)
for shape in [(4, 5), (5, 4), (2, 10), (3, 7)]:
    try:
        print(shape, "OK:\n", demo.reshape(shape))
    except ValueError as e:
        print(shape, "FAILED ->", e)

(4, 5) OK:
 [[ 1  2  3  4  5]
 [ 6  7  8  9 10]
 [11 12 13 14 15]
 [16 17 18 19 20]]
(5, 4) OK:
 [[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]
 [13 14 15 16]
 [17 18 19 20]]
(2, 10) OK:
 [[ 1  2  3  4  5  6  7  8  9 10]
 [11 12 13 14 15 16 17 18 19 20]]
(3, 7) FAILED -> cannot reshape array of size 20 into shape (3,7)


A reshape works only if **rows × columns = total elements**. 4×5, 5×4 and 2×10 all equal 20, so they work. 3×7 = 21 ≠ 20, so NumPy raises a `ValueError`: it cannot invent or drop a value.

### Task 5.4 — Combine Two Weeks

In [12]:
all_sales = np.vstack([week1, week2])
print(all_sales)
print(all_sales.shape)

[[1777  768  539]
 [1459 1048 1200]
 [ 619 1055 1465]
 [1032 1746 1685]
 [1556 1857 1581]
 [ 766 1787 1479]
 [ 647  947  748]
 [1950 1591 1879]
 [ 923 1453 1406]
 [1629  677 1272]
 [1466 1738 1484]
 [1172 1190 1008]
 [ 735  916  719]
 [ 839 1780 1288]]
(14, 3)


`vstack` stacks arrays on top of each other (adds rows): 7 + 7 = **14 rows**, columns stay 3. (`hstack` would put them side by side and give 7 × 6.)

## Block 6 — Business Analysis
### Task 6.1 — Analyze Product Categories

In [13]:
cat_total = all_sales.sum(axis=0)
cat_avg = all_sales.mean(axis=0)           # total / 14 days
cat_share = cat_total / cat_total.sum() * 100
for c, t, m, s in zip(categories, cat_total, cat_avg, cat_share):
    print(f"{c:12s} total={t:6d}  average/day={m:8.2f}  share={s:5.2f}%")
top_cat = categories[np.argmax(cat_total)]
print("Top category:", top_cat)

Electronics  total= 16570  average/day= 1183.57  share=31.34%
Clothing     total= 18553  average/day= 1325.21  share=35.09%
Home         total= 17753  average/day= 1268.07  share=33.57%
Top category: Clothing


In [14]:
print(f"Conclusion for the manager: {top_cat} brings the most revenue "
      f"({cat_share.max():.1f}% of all sales), while {categories[np.argmin(cat_total)]} is the weakest "
      f"({cat_share.min():.1f}%). Shares are close, so the mix is fairly balanced.")

Conclusion for the manager: Clothing brings the most revenue (35.1% of all sales), while Electronics is the weakest (31.3%). Shares are close, so the mix is fairly balanced.


### Task 6.2 — Find Best and Worst Days

In [15]:
day_total = all_sales.sum(axis=1)
print("Daily totals (days 1-14):", day_total)
best, worst = np.argmax(day_total), np.argmin(day_total)
print(f"Best day:  day {best+1} -> {day_total[best]}")
print(f"Worst day: day {worst+1} -> {day_total[worst]}")

Daily totals (days 1-14): [3084 3707 3139 4463 4994 4032 2342 5420 3782 3578 4688 3370 2370 3907]
Best day:  day 8 -> 5420
Worst day: day 7 -> 2342


**Business use:** the best days show what to repeat (campaigns, stock, staffing); the worst days show where to run promotions or investigate problems. Day-of-week patterns help plan inventory and staff.

### Task 6.3 — Compare Weeks

In [16]:
w1_total, w2_total = week1.sum(), week2.sum()
print("Totals:  ", w1_total, w2_total)
print("Averages (per day):", w1_total / 7, w2_total / 7)
diff = w2_total - w1_total
pct = diff / w1_total * 100
print(f"Difference (W2 - W1): {diff}")
print(f"Change: {pct:.2f}%")
print("More successful week:", "Week 2" if diff > 0 else "Week 1" if diff < 0 else "equal")

Totals:   25761 27115
Averages (per day): 3680.1428571428573 3873.5714285714284
Difference (W2 - W1): 1354
Change: 5.26%
More successful week: Week 2


### Task 6.4 — Final Analysis

In [17]:
avg_daily = day_total.mean()
below = np.where(day_total < avg_daily)[0] + 1
print("Best-selling category:", top_cat)
print("Best sales day: day", best + 1)
print("Higher average weekly revenue:", "Week 2" if w2_total > w1_total else "Week 1")
print(f"Average daily revenue: {avg_daily:.2f}")
print("Days below average:", below)

Best-selling category: Clothing
Best sales day: day 8
Higher average weekly revenue: Week 2
Average daily revenue: 3776.86
Days below average: [ 1  2  3  7 10 12 13]


In [18]:
print(f"Conclusion: over 14 days the store earned {all_sales.sum()} tenge. {top_cat} leads, "
      f"day {best+1} was the peak and day {worst+1} the low. "
      f"{'Week 2' if diff > 0 else 'Week 1'} was stronger ({pct:+.2f}% change). "
      f"{below.size} of 14 days were below the daily average ({avg_daily:.0f}); these days are candidates for promotions.")

Conclusion: over 14 days the store earned 52876 tenge. Clothing leads, day 8 was the peak and day 7 the low. Week 2 was stronger (+5.26% change). 7 of 14 days were below the daily average (3777); these days are candidates for promotions.


## Self-Check Answers
1. **Why NumPy?** Fast, loop-free operations on whole tables of numbers.
2. **axis=0 / axis=1:** axis=0 works down the rows (result per column); axis=1 works across columns (result per row).
3. **Boolean filtering vs indexing:** indexing picks by position; Boolean filtering picks by a condition on the values.
4. **Cumulative sum:** shows how a total builds up over time and when milestones are reached.
5. **vstack vs hstack:** vstack adds rows (stacks vertically), hstack adds columns (side by side).
6. **Vectorization vs loops:** shorter, clearer and much faster.
7. **Broadcasting:** lets different-shaped arrays work together, e.g. a different growth rate per category without loops.
8. **Reshape:** puts data into the shape an analysis or function needs without changing the values.

## Final Summary
I created two weeks of reproducible sales data (`seed=2026`), inspected it, selected and filtered it (slicing, Boolean masks), aggregated it (`sum`, `mean`, `axis`, `cumsum`), transformed it (vectorized +12%, `np.where` labels), reshaped and combined it (`reshape`, `-1`, `vstack`), and finished with a business analysis of categories, best/worst days, and week-over-week change. The manager conclusions are printed above (Tasks 6.1 and 6.4).